In [0]:
import os
import requests
from urllib.parse import urlparse

In [0]:
def get_token():
    auth_endpoint = f"{API_ENDPOINT}/authentication/v1/token/access-key"
    payload = {
        "accessKeyId": dbutils.secrets.get(scope="nice_credentials", key="accessKey"),
        "accessKeySecret": dbutils.secrets.get(scope="nice_credentials", key="accessSecret")
    }
    try:
        response = requests.post(auth_endpoint, json=payload)
        response.raise_for_status()
        data = response.json()
        return data["access_token"]
    except requests.exceptions.HTTPError as e:
        print(f"[HTTP Error] get_token: {e.response.status_code} - {e.response.reason}")
    return None

def get_contacts(token, start_date, end_date, max_results=10):
    url = f"{API_ENDPOINT}/incontactapi/services/v34.0/contacts/completed"
    headers = {
        "Authorization": f"Bearer {token}"
    }
    params = {
        "startDate": start_date,
        "endDate": end_date,
        "top": max_results,
        "orderBy": "lastUpdateTime DESC",
    }
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        print(f"[HTTP Error] get_contacts: {e.response.status_code} - {e.response.reason}")
    return None

def get_audio_recording(token, acd_call_id):
    """Retrieve audio recording metadata (including playback URL) for a given ACD call ID"""
    url = f"{API_ENDPOINT}/media-playback/v1/contacts"
    params = {
        "acd-call-id": acd_call_id,
        # "media-type": "all",
        # "exclude-waveforms": "true",
    }
    headers = {"Authorization": f"Bearer {token}"}
    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            return None
        else:
            return 'Error:' + str(e.response.status_code)
    return None

def download_audio(url, output_path):
    """Download audio recording"""
    response = requests.get(url, allow_redirects=True)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(response.content)
    #print(f"Audio saved to {output_path}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

In [0]:
API_ENDPOINT = "https://api-na1.niceincontact.com"
token = get_token()

In [0]:
contacts_data = get_contacts(token, "2026-03-15", "2026-03-31")

In [0]:
for contact in contacts_data["completedContacts"]:
    contact_id = contact["contactId"]
    recording_info = get_audio_recording(token, contact_id)
    #print(contact_id, recording_info)
    if recording_info is None:
        print(f"No recording found for contact ID: {contact_id}")
    # elif "playbackUrl" not in recording_info:
    #     print(f"No playback URL found for contact ID: {contact_id}")
    else:
        url = recording_info["interactions"][0]["data"]["fileToPlayUrl"]
        filename = os.path.basename(urlparse(url).path)
        output_path = f"/Workspace/Users/v578213@amerisourcebergen.com/{filename}"
        download_audio(url, output_path)